<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# From Compounding and Discounting to the Yield Curve
## A Mechanical Introduction to the Time Value of Money

&copy; Dr. Yves J. Hilpisch<br>
AI-Powered by different LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh

## Notebook Goals

This notebook reproduces the bridge's core calculations with small, transparent examples. Each section starts from familiar quantities and moves toward discount factors and the yield curve. The examples are deterministic and illustrative; they do not use market data.

You will calculate simple and natural-log returns, compare accumulation conventions, discount cash flows, price zero-coupon and coupon bonds, and derive forward rates from an illustrative zero curve. Run the cells in order and inspect the intermediate values as well as the final answers.

## 1. Simple Returns and Log Returns

A gross return is the ratio of two strictly positive prices. The simple return subtracts one from that ratio; the log return takes its natural logarithm. Rates and returns are represented as decimals in calculations, so `0.05` means five percent.

Start with a price increase from 100 to 110. Calculate all three return representations and confirm that converting the simple return with the natural logarithm reproduces the log return. `numpy.log` is the natural logarithm, conventionally written \(\ln\).

In [ ]:
import numpy as np
import pandas as pd

price_0 = 100.0
price_1 = 110.0

gross_return = price_1 / price_0
simple_return = gross_return - 1.0
log_return = np.log(gross_return)  # np.log is the natural logarithm

print(f"Gross return:  {gross_return:.4f}")
print(f"Simple return: {simple_return:.4%}")
print(f"Log return:    {log_return:.4%}")
assert np.isclose(log_return, np.log1p(simple_return))

Simple returns multiply through their gross-return factors across periods; log returns add. The price path below rises from 100 to 110 and then returns to 100. The simple returns do not sum to zero, even though the total price change is zero; the log returns do sum to zero.

In [ ]:
prices = np.array([100.0, 110.0, 100.0])
gross = prices[1:] / prices[:-1]
simple = gross - 1.0
log_returns = np.log(gross)

returns = pd.DataFrame({
    "gross_return": gross,
    "simple_return_pct": 100.0 * simple,
    "log_return_pct": 100.0 * log_returns,
})
display(returns)

print(f"Sum of simple returns: {simple.sum():.4%}")
print(f"Sum of log returns:    {log_returns.sum():.4%}")
print(f"Total price return:    {prices[-1] / prices[0] - 1:.4%}")
assert np.isclose(log_returns.sum(), np.log(prices[-1] / prices[0]))

## 2. Accumulation and Compounding

A quoted rate needs a horizon and a compounding convention to determine a value. Compare simple, annual, and continuous compounding for the same annual quote of four percent over two years. The simple formula is used only where its accumulation factor is positive.

Each row below applies a different convention to one unit of currency. The resulting factors differ even though the numerical annual quote and horizon are the same.

In [ ]:
rate = 0.04  # annual rate, stored as a decimal
years = 2.0

accumulation = pd.DataFrame(
    {"factor": [
        1.0 + rate * years,
        (1.0 + rate) ** int(years),
        np.exp(rate * years),
    ]},
    index=["Simple", "Annual compounding", "Continuous"],
)
accumulation["value_of_one"] = accumulation["factor"]
display(accumulation.round(6))

Continuous compounding is the limit of increasingly frequent discrete compounding. Here the annual rate is five percent over one year. As the number of equal periods grows, the discrete accumulation factor approaches the exponential factor. The period count is an integer by construction.

In [ ]:
rate = 0.05
years = 1.0
periods = np.array([1, 4, 12, 52, 365], dtype=int)

discrete_factors = (1.0 + rate * years / periods) ** periods
continuous_factor = np.exp(rate * years)

limit_table = pd.DataFrame({
    "periods_per_year": periods,
    "discrete_factor": discrete_factors,
})
display(limit_table.round(6))
print(f"Continuous factor: {continuous_factor:.6f}")
assert np.all(np.diff(discrete_factors) > 0.0)
assert discrete_factors[-1] < continuous_factor

## 3. Discounting and Zero-Coupon Bonds

Discounting reverses accumulation: under continuous compounding, the discount factor is the exponential of minus the rate times the year fraction. A default-free zero-coupon bond pays its face value once at maturity, so its price is face value multiplied by that maturity's discount factor.

Discount 100 currency units due in two years at a continuously compounded zero rate of four percent. Accumulating the resulting present value at the same rate should recover the original payment, up to floating-point rounding.

In [ ]:
face_value = 100.0
maturity = 2.0  # year fraction; maturities need not be integers
zero_rate = 0.04  # continuously compounded annual rate

discount_factor = np.exp(-zero_rate * maturity)
zero_bond_price = face_value * discount_factor
reinvested_value = zero_bond_price * np.exp(zero_rate * maturity)

print(f"Discount factor: {discount_factor:.6f}")
print(f"Zero-coupon bond price: {zero_bond_price:.4f}")
print(f"Value at maturity: {reinvested_value:.4f}")
assert np.isclose(reinvested_value, face_value)

A coupon bond is a collection of dated cash flows. To isolate that arithmetic, use the illustrative discount factors from the slides: 0.96 at year 1 and 0.91 at year 2. The example is synthetic, not a market quote. Discount each coupon and the final coupon-plus-principal separately, then sum the present values.

In [ ]:
cashflow_times = np.array([1.0, 2.0])
cashflows = np.array([5.0, 105.0])  # final coupon plus face value
illustrative_discounts = np.array([0.96, 0.91])

present_values = cashflows * illustrative_discounts
cashflow_table = pd.DataFrame({
    "time_years": cashflow_times,
    "cashflow": cashflows,
    "discount_factor": illustrative_discounts,
    "present_value": present_values,
})
display(cashflow_table)
bond_price = present_values.sum()
print(f"Illustrative coupon-bond price: {bond_price:.2f}")
assert np.isclose(bond_price, 5.0 * 0.96 + 105.0 * 0.91)

## 4. From Zero Rates to Discount and Forward Curves

A zero rate at maturity \(T\) gives the discount factor for a unit payment at \(T\). Under continuous compounding, \(D(0,T)=e^{-z(0,T)T}\). Two discount factors determine the accumulation factor, and hence the continuously compounded forward rate, between their maturities.

Build the illustrative curve used in the article and slides. Maturities are year fractions and may be non-integers. The zero rates are entered as decimals; only the presentation columns convert them to percentages. The first forward rate is undefined because there is no earlier curve node.

In [ ]:
maturity = np.array([0.5, 1.0, 2.0, 5.0], dtype=float)
zero_rate = np.array([0.035, 0.037, 0.039, 0.042])

discount = np.exp(-zero_rate * maturity)
forward = np.diff(zero_rate * maturity) / np.diff(maturity)

curve = pd.DataFrame({
    "maturity_years": maturity,
    "zero_rate_pct": 100.0 * zero_rate,
    "discount_factor": discount,
    "next_interval_forward_pct": np.r_[np.nan, 100.0 * forward],
})
display(curve.round(6))

Check both transformations numerically. First recover zero rates from discount factors. Then use each forward rate to reconstruct the next discount factor from the previous one. These checks catch common sign, indexing, and unit errors; they do not establish that an assumed curve is a forecast.

In [ ]:
recovered_zero_rate = -np.log(discount) / maturity
rebuilt_discount = discount[:-1] * np.exp(-forward * np.diff(maturity))

print("Recovered zero rates (%):", 100.0 * recovered_zero_rate)
print("Discount factors rebuilt:", rebuilt_discount)
print("Original next factors:   ", discount[1:])

assert np.allclose(recovered_zero_rate, zero_rate)
assert np.allclose(rebuilt_discount, discount[1:])

## 5. What These Calculations Do—and Do Not—Cover

The calculations here move from returns to accumulation, discounting, zero-coupon bond prices, and forward rates using a given deterministic curve. They leave market conventions, curve bootstrapping, interpolation choices, credit risk, and stochastic interest-rate models for the rates specialization. The identities are useful foundations, but real instruments require additional market-specific definitions.

For further study, see the companion bridge note and *Python & AI for Rates, Bonds, and Credit*: Chapter 2 develops cash flows and present value; Chapter 4 develops zero and forward curves; Chapter 3 adds bond conventions; and Chapter 6 introduces curve construction.

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
